In [37]:
import pyrealsense2 as rs, json, cv2, numpy as np, os

bag_path = r"C:\Users\admin\Documents\Vicki\School stuff\SEGP\FFB Samples\FFB10\ffb10Depth_3D.bag"
out_dir = "sample_001"; os.makedirs(out_dir, exist_ok=True)

pipe = rs.pipeline()
cfg = rs.config(); cfg.enable_device_from_file(bag_path, repeat_playback=False)
cfg.enable_stream(rs.stream.color); cfg.enable_stream(rs.stream.depth)
profile = pipe.start(cfg)

align = rs.align(rs.stream.color)
depth_sensor = profile.get_device().first_depth_sensor()
depth_scale = depth_sensor.get_depth_scale()

color_stream = profile.get_stream(rs.stream.color).as_video_stream_profile()
intr = color_stream.get_intrinsics()  # fx, fy, ppx(cx), ppy(cy), width, height

with open(os.path.join(out_dir, "intrinsics.json"), "w") as f:
    json.dump({"fx":intr.fx,"fy":intr.fy,"cx":intr.ppx,"cy":intr.ppy,
               "width":intr.width,"height":intr.height,"depth_scale":depth_scale}, f, indent=2)

i=0
try:
    while True:
        frames = pipe.wait_for_frames()
        frames = align.process(frames)
        depth = frames.get_depth_frame()
        color = frames.get_color_frame()
        if not depth or not color: continue
        i += 1
        rgb = np.asanyarray(color.get_data())
        dep = np.asanyarray(depth.get_data())  # uint16 in depth units
        cv2.imwrite(os.path.join(out_dir, f"rgb_{i:04d}.png"), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
        cv2.imwrite(os.path.join(out_dir, f"depth_{i:04d}.png"), dep)
except Exception:
    pass
finally:
    pipe.stop()


In [40]:
import cv2, numpy as np, json

meta = json.load(open("sample_001/intrinsics.json"))
scale = meta["depth_scale"]

D = cv2.imread("sample_001/depth_0001.png", cv2.IMREAD_UNCHANGED)
valid = D > 0
print("depth_scale:", scale)
print("valid %:", 100*valid.mean())
vals_m = (D[valid] * scale).astype(float)
print("valid depth (m): min", vals_m.min(), "med", np.median(vals_m), "max", vals_m.max())


depth_scale: 0.0010000000474974513
valid %: 92.16069878472221
valid depth (m): min 0.5420000257436186 med 1.5340000728610903 max 11.710000556195155


Object detection

In [45]:
import os, glob, requests, time

print("cwd:", os.getcwd())
print("sample_001 exists?", os.path.isdir("sample_001"))
rgb_files = sorted(glob.glob("sample_001/rgb_*.png"))
print("rgb files found:", len(rgb_files))
print("first 3:", rgb_files[:3])

# quick internet check (should return some status quickly)
try:
    r = requests.get("https://serverless.roboflow.com", timeout=8)
    print("Roboflow server reachable:", r.status_code)
except Exception as e:
    print("Network error:", e)


cwd: C:\Users\admin\Documents\Vicki\School stuff\SEGP
sample_001 exists? True
rgb files found: 43
first 3: ['sample_001\\rgb_0001.png', 'sample_001\\rgb_0002.png', 'sample_001\\rgb_0003.png']
Roboflow server reachable: 200


In [50]:
from inference_sdk import InferenceHTTPClient
import json

API_KEY   = "2nJlJr8fFV12sUCfK54X"
WORKSPACE = "ffbdetection-sumlf"
WORKFLOW  = "find-ffbs"

client = InferenceHTTPClient(api_url="https://serverless.roboflow.com", api_key=API_KEY)
res = client.run_workflow(
    workspace_name=WORKSPACE,
    workflow_id=WORKFLOW,
    images={"image": r"C:\Users\admin\Documents\Vicki\School stuff\SEGP\sample_001\rgb_0001.png"},
    use_cache=True
)

# print a trimmed preview so we can see the shape
def preview(x, maxlen=1200):
    s = json.dumps(x, indent=2) if not isinstance(x, str) else x
    print(s[:maxlen] + ("..." if len(s) > maxlen else ""))

preview(res)


[
  {
    "predictions": {
      "image": {
        "width": 1280,
        "height": 720
      },
      "predictions": [
        {
          "width": 108.802978515625,
          "height": 168.796875,
          "x": 1225.5985107421875,
          "y": 533.353271484375,
          "confidence": 0.9660595655441284,
          "class_id": 0,
          "class": "ffb",
          "detection_id": "f978d5d0-c2d9-4a4c-a733-9cee1d52db21",
          "parent_id": "image"
        },
        {
          "width": 224.8236083984375,
          "height": 181.19949340820312,
          "x": 651.2186889648438,
          "y": 376.9242706298828,
          "confidence": 0.9599176645278931,
          "class_id": 0,
          "class": "ffb",
          "detection_id": "e6b8f868-a1e1-47af-8867-be9a09526dca",
          "parent_id": "image"
        },
        {
          "width": 63.911376953125,
          "height": 40.82679271697998,
          "x": 1247.6949462890625,
          "y": 35.038870334625244,
          "conf

In [52]:
from inference_sdk import InferenceHTTPClient
import cv2, glob, csv, os, time

API_KEY   = "2nJlJr8fFV12sUCfK54X"
WORKSPACE = "ffbdetection-sumlf"
WORKFLOW  = "find-ffbs"

# Use your absolute frames directory if needed
frames_dir  = r"C:\Users\admin\Documents\Vicki\School stuff\SEGP\sample_001"
out_vis_dir = os.path.join(frames_dir, "detections_vis_rf")
os.makedirs(out_vis_dir, exist_ok=True)
csv_path = os.path.join(frames_dir, "detections_rf.csv")

client = InferenceHTTPClient(api_url="https://serverless.roboflow.com", api_key=API_KEY)

def run_wf(img_path):
    return client.run_workflow(
        workspace_name=WORKSPACE,
        workflow_id=WORKFLOW,
        images={"image": img_path},
        use_cache=True
    )

def get_preds(res):
    """RAPID shape: [ { 'predictions': { 'image': {...}, 'predictions': [ ... ] } } ]"""
    if isinstance(res, list) and res:
        top = res[0]
        if isinstance(top, dict) and "predictions" in top:
            inner = top["predictions"]
            if isinstance(inner, dict) and "predictions" in inner:
                return inner["predictions"], inner.get("image", {})
    return [], {}

# ---- process all frames ----
rgb_files = sorted(glob.glob(os.path.join(frames_dir, "rgb_*.png")))
print("Found frames:", len(rgb_files))

with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["frame","x1","y1","x2","y2","conf","class"])

    for rf in rgb_files:
        base = os.path.basename(rf)
        try:
            t0 = time.time()
            res = run_wf(rf)
            preds, info = get_preds(res)
            print(f"{base}: {len(preds)} preds in {time.time()-t0:.2f}s")

            # optional confidence floor to ignore spurious boxes
            preds = [p for p in preds if float(p.get("confidence", 0)) >= 0.25]

            if not preds:
                w.writerow([base,"","","","","",""])
                continue

            # --- choose detection nearest to image center, prefer central window ---
            # get image size (from metadata or read image)
            W = int(info.get("width", 0)); H = int(info.get("height", 0))
            if W == 0 or H == 0:
                tmp = cv2.imread(rf); H, W = tmp.shape[:2]

            cx0, cy0 = W / 2.0, H / 2.0

            # central window size (tweak if needed)
            CENTER_WIN_W = 0.40   # 40% of width around center
            CENTER_WIN_H = 0.40   # 40% of height around center

            x_left   = cx0 - (W * CENTER_WIN_W) / 2.0
            x_right  = cx0 + (W * CENTER_WIN_W) / 2.0
            y_top    = cy0 - (H * CENTER_WIN_H) / 2.0
            y_bottom = cy0 + (H * CENTER_WIN_H) / 2.0

            def in_center_window(p):
                return (x_left <= float(p["x"]) <= x_right) and (y_top <= float(p["y"]) <= y_bottom)

            def center_dist2(p):
                dx = float(p["x"]) - cx0
                dy = float(p["y"]) - cy0
                return dx*dx + dy*dy

            # prefer predictions inside the central window
            central = [p for p in preds if in_center_window(p)]
            cands = central if central else preds

            # choose nearest to center; tie-break by higher confidence
            best = min(cands, key=lambda p: (center_dist2(p), -float(p.get("confidence", 0.0))))

            # convert center-wh to corners
            cx, cy = float(best["x"]), float(best["y"])
            ww, hh = float(best["width"]), float(best["height"])
            x1, y1 = int(cx - ww/2), int(cy - hh/2)
            x2, y2 = int(cx + ww/2), int(cy + hh/2)
            conf   = float(best.get("confidence", 0.0))
            label  = best.get("class", "ffb")

            # draw & save
            img = cv2.imread(rf)
            cv2.rectangle(img, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(img, f"{label} {conf:.2f}", (x1, max(0,y1-5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
            cv2.imwrite(os.path.join(out_vis_dir, base), img)

            # write CSV
            w.writerow([base, x1,y1,x2,y2, round(conf,4), label])

        except Exception as e:
            print("ERROR on", base, "->", repr(e))


Found frames: 43
rgb_0001.png: 5 preds in 43.86s
rgb_0002.png: 5 preds in 47.99s
rgb_0003.png: 6 preds in 48.45s
rgb_0004.png: 6 preds in 55.40s
rgb_0005.png: 6 preds in 53.42s
rgb_0006.png: 5 preds in 66.18s
rgb_0007.png: 5 preds in 42.21s
rgb_0008.png: 4 preds in 38.61s
rgb_0009.png: 4 preds in 42.16s
rgb_0010.png: 4 preds in 32.36s
rgb_0011.png: 4 preds in 8.36s
rgb_0012.png: 4 preds in 10.04s
rgb_0013.png: 4 preds in 55.06s
rgb_0014.png: 4 preds in 53.78s
rgb_0015.png: 4 preds in 38.97s
rgb_0016.png: 4 preds in 49.25s
rgb_0017.png: 4 preds in 50.32s
rgb_0018.png: 4 preds in 63.94s
rgb_0019.png: 5 preds in 52.01s
rgb_0020.png: 5 preds in 16.57s
rgb_0021.png: 4 preds in 19.28s
rgb_0022.png: 4 preds in 49.50s
rgb_0023.png: 4 preds in 41.06s
rgb_0024.png: 4 preds in 48.29s
rgb_0025.png: 5 preds in 50.20s
rgb_0026.png: 5 preds in 51.14s
rgb_0027.png: 3 preds in 66.17s
rgb_0028.png: 3 preds in 58.30s
rgb_0029.png: 3 preds in 55.44s
rgb_0030.png: 3 preds in 44.67s
rgb_0031.png: 5 preds in

Masking


In [3]:
import os, csv, json, glob, cv2, numpy as np

frames_dir = r"C:\Users\admin\Documents\Vicki\School stuff\SEGP\sample_001"
det_csv    = os.path.join(frames_dir, "detections_rf.csv")
intr_path  = os.path.join(frames_dir, "intrinsics.json")

out_mask_dir = os.path.join(frames_dir, "masks")
out_vis_dir  = os.path.join(frames_dir, "masks_vis")
os.makedirs(out_mask_dir, exist_ok=True)
os.makedirs(out_vis_dir,  exist_ok=True)

meta  = json.load(open(intr_path))
scale = float(meta["depth_scale"])  # meters per depth unit

def depth_band_mask(depth_u16, bbox, scale, band_cm=8):
    x1,y1,x2,y2 = map(int, bbox)
    h, w = depth_u16.shape
    x1 = max(0, min(w-1, x1)); x2 = max(0, min(w-1, x2))
    y1 = max(0, min(h-1, y1)); y2 = max(0, min(h-1, y2))
    if x2<=x1 or y2<=y1:
        return np.zeros_like(depth_u16, np.uint8)

    crop = depth_u16[y1:y2, x1:x2]
    z = crop[crop>0]
    if z.size == 0:
        return np.zeros_like(depth_u16, np.uint8)

    z_med = np.median(z)
    band  = int((band_cm/100.0) / scale)   # ± band_cm around median depth
    m_roi = ((depth_u16 >= (z_med - band)) &
             (depth_u16 <= (z_med + band))).astype(np.uint8) * 255

    mask = np.zeros_like(depth_u16, np.uint8)
    mask[y1:y2, x1:x2] = m_roi[y1:y2, x1:x2]
    return mask

# Map RGB filename -> depth filename
depth_files = {os.path.basename(p): p for p in glob.glob(os.path.join(frames_dir, "depth_*.png"))}
rgb_to_depth = lambda rgb_name: depth_files.get(rgb_name.replace("rgb_", "depth_"))

with open(det_csv, newline="") as f:
    r = csv.DictReader(f)
    for row in r:
        name = row["frame"]
        if not row["x1"]:    # no detection
            continue
        x1,y1,x2,y2 = map(float, [row["x1"], row["y1"], row["x2"], row["y2"]])
        depth_path = rgb_to_depth(name)
        rgb_path   = os.path.join(frames_dir, name)
        if not depth_path or not os.path.exists(depth_path):
            continue

        D = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
        mask = depth_band_mask(D, (x1,y1,x2,y2), scale, band_cm=10) #tweak this

        mask_path = os.path.join(out_mask_dir, name.replace("rgb_", "mask_"))
        cv2.imwrite(mask_path, mask)

        rgb = cv2.imread(rgb_path)
        overlay = rgb.copy()
        overlay[mask==0] = (overlay[mask==0] * 0.25).astype(np.uint8)  # dim background
        vis_path = os.path.join(out_vis_dir, name)
        cv2.imwrite(vis_path, overlay)

print("✅ Masks saved to:", out_mask_dir)
print("✅ Visual previews saved to:", out_vis_dir)


✅ Masks saved to: C:\Users\admin\Documents\Vicki\School stuff\SEGP\sample_001\masks
✅ Visual previews saved to: C:\Users\admin\Documents\Vicki\School stuff\SEGP\sample_001\masks_vis


3D Reconstruction


In [ ]:
import os, glob, json, numpy as np, cv2, open3d as o3d

frames_dir = r"C:\Users\admin\Documents\Vicki\School stuff\SEGP\sample_001"
intr_p     = os.path.join(frames_dir, "intrinsics.json")
mask_dir   = os.path.join(frames_dir, "masks")

# --- intrinsics & scale ---
meta = json.load(open(intr_p))
fx, fy, cx, cy = meta["fx"], meta["fy"], meta["cx"], meta["cy"]
W, H = meta["width"], meta["height"]
depth_scale_units_per_m = 1.0 / float(meta["depth_scale"])

# --- downscale factor to cut memory (0.5 = half width/height) ---
S = 0.5
W2, H2 = int(W*S), int(H*S)
fx2, fy2, cx2, cy2 = fx*S, fy*S, cx*S, cy*S
intr = o3d.camera.PinholeCameraIntrinsic(W2, H2, fx2, fy2, cx2, cy2)

# --- list frames & limit count for a first run ---
rgb_files = sorted(glob.glob(os.path.join(frames_dir, "rgb_*.png")))
dep_files = [os.path.join(frames_dir, os.path.basename(p).replace("rgb_", "depth_")) for p in rgb_files]
msk_files = [os.path.join(mask_dir,     os.path.basename(p).replace("rgb_", "mask_"))  for p in rgb_files]
MAX_FRAMES = 20  # try 20 first; increase later
rgb_files, dep_files, msk_files = rgb_files[:MAX_FRAMES], dep_files[:MAX_FRAMES], msk_files[:MAX_FRAMES]

def rgbd_from_paths(rgb_path, depth_path, mask_path):
    # read & downscale color
    color_cv = cv2.imread(rgb_path, cv2.IMREAD_COLOR)
    color_cv = cv2.resize(color_cv, (W2, H2), interpolation=cv2.INTER_AREA)
    color = o3d.geometry.Image(cv2.cvtColor(color_cv, cv2.COLOR_BGR2RGB))

    # read depth (uint16), apply mask, downscale with nearest to keep units
    depth_cv = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
    if os.path.exists(mask_path):
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        depth_cv[m==0] = 0
    depth_cv = cv2.resize(depth_cv, (W2, H2), interpolation=cv2.INTER_NEAREST)
    depth = o3d.geometry.Image(depth_cv)

    return o3d.geometry.RGBDImage.create_from_color_and_depth(
        color, depth,
        depth_scale=depth_scale_units_per_m,   # e.g., 1000 for meters
        depth_trunc=3.0,                       # tighter truncation reduces memory
        convert_rgb_to_intensity=False
    )

# --- Coarser TSDF to save memory ---
tsdf = o3d.pipelines.integration.ScalableTSDFVolume(
    voxel_length=0.010,   # 10 mm voxels (start coarse; lower later e.g., 0.006 or 0.004)
    sdf_trunc=0.04,       # 4 cm truncation
    color_type=o3d.pipelines.integration.TSDFVolumeColorType.RGB8
)

# --- Integrate with identity poses (no odometry) ---
T = np.eye(4)
for rf, df, mf in zip(rgb_files, dep_files, msk_files):
    if not (os.path.exists(rf) and os.path.exists(df) and os.path.exists(mf)):
        continue
    rgbd = rgbd_from_paths(rf, df, mf)
    tsdf.integrate(rgbd, intr, T)

mesh = tsdf.extract_triangle_mesh()
mesh.remove_degenerate_triangles()
mesh.remove_duplicated_triangles()
mesh.remove_duplicated_vertices()
mesh.remove_non_manifold_edges()
mesh.compute_vertex_normals()

mesh_path = os.path.join(frames_dir, "ffb_mesh_coarse.ply")
o3d.io.write_triangle_mesh(mesh_path, mesh)
print("✅ Mesh saved:", mesh_path)

# quick size sanity-check
aabb = mesh.get_axis_aligned_bounding_box()
print("AABB extents (m):", aabb.get_extent())
